# ルオスエン映像制作｜Colab GPU レンダー環境（Blender Cycles）
**DM専用・ネタバレあり。** 第一話エンディング『空に降る雪』の3D制作をColab GPUで行うための共通ノートブック。

- 正本：GitHub `Makeinu1/DaD_Ruosuen`（このノートは毎回 main を取得して実行）
- 出力：Google Drive `MyDrive/Ruosuen_Render/`（素材受け渡し場所を兼ねる）
- **ランタイムは「L4 GPU」を選ぶ**（ランタイム → ランタイムのタイプを変更 → L4）。A100/H100はユニット消費が数倍なので最終本番以外は使わない。
- 使い終わったら最後のセルで**必ずランタイムを切断**（つけっぱなしでもユニットが減る）。

| セル | 内容 | 毎回 |
|---|---|---|
| 1 | 設定 | ✓ |
| 2 | GPU確認・Drive接続 | ✓ |
| 3 | GitHubから最新取得 | ✓ |
| 4 | Blender導入（Driveにキャッシュ、2回目以降は約30秒） | ✓ |
| 5 | GPUレンダー動作テスト | 初回・不調時 |
| 6 | レンダー実行 | ✓ |
| 7 | 連番→MP4（任意） | 動画時 |
| 8 | ランタイム切断 | ✓ |


In [ ]:
# @title 1. 設定（ここだけ編集）
REPO_URL   = "https://github.com/Makeinu1/DaD_Ruosuen"
BRANCH     = "main"            # @param {type:"string"}
PIPELINE   = "dm/episode-01/ending/video-pipeline/v7_blender_cycles"
# 実行するBlenderスクリプト（PIPELINE/scripts/ からの相対パス、またはリポジトリ直下からのパス）
RENDER_SCRIPT = "scripts/render_step5_lookdev_cycles.py"  # @param {type:"string"}
BLENDER_VERSION = "5.0.1"      # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/Ruosuen_Render"
AUTO_DISCONNECT = True         # @param {type:"boolean"}  # セル6完了後に自動で切断してユニット節約


In [ ]:
# @title 2. GPU確認・Drive接続
import subprocess, os, datetime
try:
    print(subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"]).decode())
except Exception:
    raise SystemExit("GPUが割り当てられていません。ランタイム → ランタイムのタイプを変更 → L4 GPU を選んで再接続してください。")
from google.colab import drive
drive.mount("/content/drive")
for d in ["", "cache", "renders", "logs"]:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)
RUN_ID = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime("%Y%m%d-%H%M")
print("RUN_ID:", RUN_ID, "| 出力先:", DRIVE_ROOT)


In [ ]:
# @title 3. GitHubから最新を取得（公開リポジトリ・読み取りのみ）
import shutil
REPO_DIR = "/content/DaD_Ruosuen"
if os.path.exists(REPO_DIR):
    subprocess.run(["git","-C",REPO_DIR,"fetch","--depth","1","origin",BRANCH], check=True)
    subprocess.run(["git","-C",REPO_DIR,"checkout","-q","FETCH_HEAD"], check=True)
else:
    subprocess.run(["git","clone","-q","--depth","1","--branch",BRANCH,REPO_URL,REPO_DIR], check=True)
COMMIT = subprocess.check_output(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"]).decode().strip()
PIPE_DIR = os.path.join(REPO_DIR, PIPELINE)
print("commit:", COMMIT); print(sorted(os.listdir(os.path.join(PIPE_DIR,"scripts"))))


In [ ]:
# @title 4. Blender導入（DriveにキャッシュするのでDL は初回のみ）
BL_DIR = "/content/blender"
major = ".".join(BLENDER_VERSION.split(".")[:2])
tar_name = f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
cache_tar = os.path.join(DRIVE_ROOT, "cache", tar_name)
if not os.path.exists(f"{BL_DIR}/blender"):
    if not os.path.exists(cache_tar):
        url = f"https://download.blender.org/release/Blender{major}/{tar_name}"
        print("初回ダウンロード:", url)
        subprocess.run(["curl","-fSL",url,"-o","/content/bl.tar.xz"], check=True)
        shutil.copy("/content/bl.tar.xz", cache_tar)
    else:
        shutil.copy(cache_tar, "/content/bl.tar.xz")
    os.makedirs(BL_DIR, exist_ok=True)
    subprocess.run(["tar","-xf","/content/bl.tar.xz","-C",BL_DIR,"--strip-components=1"], check=True)
    os.remove("/content/bl.tar.xz")
# Blender実行に必要な共有ライブラリ
subprocess.run("apt-get -qq install -y libxi6 libxkbcommon0 libxrender1 libsm6 libgl1 > /dev/null 2>&1", shell=True)
BLENDER = f"{BL_DIR}/blender"
print(subprocess.check_output([BLENDER,"--version"]).decode().splitlines()[0])


In [ ]:
# @title 5. GPUレンダー動作テスト（約1分・OptiXが使えるか確認）
test_py = r'''
import bpy, time
sc = bpy.context.scene
sc.render.engine = "CYCLES"
p = bpy.context.preferences.addons["cycles"].preferences
for be in ("OPTIX","CUDA"):
    try:
        p.compute_device_type = be; p.get_devices()
        g = [d for d in p.devices if d.type != "CPU"]
        if g:
            for d in p.devices: d.use = d.type != "CPU"
            sc.cycles.device = "GPU"; print("[GPU]", be, [d.name for d in g]); break
    except Exception as e: print("skip", be, e)
sc.cycles.samples = 128
sc.render.resolution_x, sc.render.resolution_y = 1920, 1080
sc.render.filepath = "/content/gpu_test.png"
t = time.time(); bpy.ops.render.render(write_still=True); print(f"[TEST] 1080p 128spp: {time.time()-t:.1f}s")
'''
open("/content/gpu_test.py","w").write(test_py)
r = subprocess.run([BLENDER,"-b","--factory-startup","--python","/content/gpu_test.py"], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("[GPU]","[TEST]","skip"))) or r.stdout[-2000:] + r.stderr[-2000:])


In [ ]:
# @title 6. レンダー実行 → Driveへ保存
import glob, time
WORK = "/content/work"
shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
for sub in ("assets","textures"):
    shutil.copytree(os.path.join(PIPE_DIR, sub), os.path.join(WORK, sub))
script = os.path.join(PIPE_DIR, RENDER_SCRIPT) if not RENDER_SCRIPT.startswith("dm/") else os.path.join(REPO_DIR, RENDER_SCRIPT)
env = dict(os.environ, V7_WORK_DIR=WORK, RUOSUEN_OUT=WORK)
name = os.path.splitext(os.path.basename(script))[0]
log_path = os.path.join(DRIVE_ROOT, "logs", f"{RUN_ID}_{name}.log")
t0 = time.time()
with open(log_path, "w") as log:
    p = subprocess.Popen([BLENDER,"-b","--factory-startup","--python",script], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line)
        if line.startswith(("[GPU]","[STEP","[FLOWERS]","Saved","Error","Traceback")) or "Fra:" in line and "Sample" not in line:
            print(line.rstrip())
    p.wait()
print(f"exit={p.returncode}  {time.time()-t0:.0f}s  log: {log_path}")
dst = os.path.join(DRIVE_ROOT, "renders", f"{RUN_ID}_{name}_{COMMIT}")
os.makedirs(dst, exist_ok=True)
n = 0
for d in glob.glob(os.path.join(WORK, "out*")):
    for f in glob.glob(os.path.join(d, "**", "*"), recursive=True):
        if os.path.isfile(f):
            rel = os.path.relpath(f, WORK); os.makedirs(os.path.dirname(os.path.join(dst, rel)), exist_ok=True)
            shutil.copy(f, os.path.join(dst, rel)); n += 1
print(f"{n} ファイルを保存: {dst}")
if p.returncode != 0:
    raise RuntimeError("レンダー失敗。ログを確認してください: " + log_path)


In [ ]:
# @title 7. 連番PNG → MP4（任意：動画ショットを出したとき）
FRAMES_GLOB = ""   # @param {type:"string"}  # 例: /content/work/out_anim/n08_%04d.png
FPS = 24           # @param {type:"integer"}
if FRAMES_GLOB:
    out_mp4 = os.path.join(dst, f"{name}.mp4")
    # L4のNVENCで高速エンコード。プロジェクター上映用に yuv420p / Rec.709
    cmd = ["ffmpeg","-y","-framerate",str(FPS),"-i",FRAMES_GLOB,"-c:v","h264_nvenc","-preset","p5","-cq","18",
           "-pix_fmt","yuv420p","-colorspace","bt709","-color_primaries","bt709","-color_trc","bt709",out_mp4]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:  # NVENC不可ならCPUエンコード
        cmd[cmd.index("h264_nvenc")] = "libx264"; cmd[cmd.index("-preset")+1] = "veryfast"
        cmd[cmd.index("-cq")] = "-crf"
        subprocess.run(cmd, check=True)
    print("MP4:", out_mp4)


In [ ]:
# @title 8. ランタイム切断（コンピューティングユニット節約）
if AUTO_DISCONNECT:
    from google.colab import runtime
    print("切断します。結果は Drive の Ruosuen_Render/ にあります。")
    runtime.unassign()
